# BackdoorBench Low Frequency 공격 실험 (Colab 실행용)

이 노트북은 Google Colab에서 순서대로 실행하는 파일입니다.  
세션이 끊길 수 있으므로 **Drive 마운트, 백업, 복원 셀**을 먼저 실행합니다.


## 셀 1. Drive 마운트와 경로 설정


In [ ]:
# Colab 런타임이 끊겨도 결과를 보존하기 위해 Google Drive를 먼저 연결합니다.
from google.colab import drive
drive.mount('/content/drive')

import os

# 실험 이름은 BackdoorBench의 record 폴더명과 맞춥니다.
PROJECT_NAME = "BackdoorBench"
EXP_NAME = "lf_0_1"
DRIVE_ROOT = "/content/drive/MyDrive/BackdoorBench_results"
DRIVE_EXP = f"{DRIVE_ROOT}/{EXP_NAME}"

# Drive에 결과 백업 폴더를 미리 만들어 둡니다.
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DRIVE_EXP, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Experiment backup:", DRIVE_EXP)


## 셀 2. GPU와 런타임 확인


In [ ]:
# GPU가 잡혔는지 확인합니다. CUDA available이 True여야 정상입니다.
!nvidia-smi

import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 셀 3. BackdoorBench 클론과 폴더 초기화


In [ ]:
# GitHub 원본 BackdoorBench를 Colab 로컬 디스크에 받습니다.
import os

%cd /content

if not os.path.exists("/content/BackdoorBench"):
    !git clone https://github.com/SCLBD/BackdoorBench.git

%cd /content/BackdoorBench

# record, data 등 실행에 필요한 기본 폴더를 생성합니다.
!sh ./sh/init_folders.sh


## 셀 4. 의존성 설치


In [ ]:
# Colab 기본 PyTorch는 그대로 쓰고, BackdoorBench와 시각화에 필요한 패키지만 설치합니다.
!pip install -q opencv-python pandas Pillow scikit-learn scikit-image tqdm pyyaml \
    tensorboard kornia imageio matplotlib scipy seaborn \
    shap grad-cam plotly umap-learn graphviz hiddenlayer \
    PyHessian "torchmetrics[image]" pytorch-wavelets

# pytorchviz는 GitHub 최신 버전을 설치합니다.
!pip install -q -U git+https://github.com/szagoruyko/pytorchviz.git@master


## 셀 5. Colab 호환성 패치


In [ ]:
# BackdoorBench 일부 코드는 오래된 PyTorch/NumPy 기준이라 Colab 최신 환경에서 패치가 필요합니다.
%cd /content/BackdoorBench

# PyTorch 2.6+ torch.load 기본값 변화 대응
!sed -i 's/torch.load(save_path)/torch.load(save_path, weights_only=False)/g' ./utils/save_load_attack.py
!sed -i 's/torch.load(load_path)/torch.load(load_path, weights_only=False)/g' ./utils/save_load_attack.py

# NumPy 2.x에서 제거된 np.infty, np.float alias를 변경
!sed -i 's/np.infty/np.inf/g' ./utils/trainer_cls.py
!sed -i 's/np.float/float/g' ./utils/bd_img_transform/patch.py

# grad-cam 관련 use_cuda 인자가 남아 있는지 확인합니다. 출력만 확인하면 됩니다.
!grep -R "use_cuda" -n ./analysis ./utils || true


## 셀 6. 백업/복원 함수 정의


In [ ]:
# 공격/방어가 끝날 때마다 backup_record()를 호출하면 Drive에 결과가 저장됩니다.
import os, shutil, glob, json, subprocess, textwrap

EXP_NAME = "lf_0_1"
DRIVE_ROOT = "/content/drive/MyDrive/BackdoorBench_results"
DRIVE_EXP = f"{DRIVE_ROOT}/{EXP_NAME}"

def backup_record():
    """현재 /content의 record 결과를 Drive로 복사합니다."""
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    src = f"/content/BackdoorBench/record/{EXP_NAME}"
    dst = DRIVE_EXP
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Backed up: {src} -> {dst}")
    else:
        print(f"Not found: {src}")

def restore_record():
    """Drive에 저장된 결과를 Colab 로컬 record 폴더로 복원합니다."""
    src = DRIVE_EXP
    dst = f"/content/BackdoorBench/record/{EXP_NAME}"
    if os.path.exists(src):
        os.makedirs("/content/BackdoorBench/record", exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Restored: {src} -> {dst}")
    else:
        print(f"No backup found: {src}")


## 셀 7. 세션 재시작 시 복원


In [ ]:
# 처음 실행이면 백업이 없을 수 있습니다. 세션이 끊긴 뒤 재개할 때는 이 셀을 실행합니다.
%cd /content/BackdoorBench
restore_record()
!ls -lah ./record/lf_0_1 || true


## 셀 8. Low Frequency 공격 실행


In [ ]:
# LF 공격을 실행합니다. 결과는 ./record/lf_0_1에 저장됩니다.
%cd /content/BackdoorBench

!python ./attack/lf.py \
    --yaml_path ./config/attack/prototype/cifar10.yaml \
    --save_folder_name lf_0_1

# 공격이 끝나면 바로 Drive에 백업합니다.
backup_record()


## 셀 9. 공격 결과 확인


In [ ]:
# attack_df.csv에서 최종 epoch의 ACC, ASR, RA를 확인합니다.
import pandas as pd, os

base = "./record/lf_0_1"
summary_path = f"{base}/attack_df_summary.csv"
df_path = f"{base}/attack_df.csv"

if os.path.exists(summary_path):
    print("[attack_df_summary.csv]")
    display(pd.read_csv(summary_path))

df = pd.read_csv(df_path)
last = df.iloc[-1]

print("\n[Final epoch]")
for c in ["epoch", "train_acc", "test_acc", "test_asr", "test_ra", "train_epoch_loss_avg_over_batch"]:
    if c in df.columns:
        print(f"{c:40s}: {last[c]}")


## 셀 10. 학습 곡선 출력


In [ ]:
# BackdoorBench가 저장한 loss/accuracy 계열 그래프를 Colab에 표시합니다.
from IPython.display import Image, display
import os

for img in ["loss_metric_plots.png", "acc_like_metric_plots.png"]:
    path = f"./record/lf_0_1/{img}"
    if os.path.exists(path):
        print(path)
        display(Image(path))
    else:
        print("Not found:", path)


## 셀 11. clean vs LF backdoor 이미지 비교


In [ ]:
# clean 이미지, LF backdoor 이미지, 두 이미지의 차이맵을 나란히 저장합니다.
%cd /content/BackdoorBench

import torch, numpy as np, matplotlib.pyplot as plt, os

result = torch.load("./record/lf_0_1/attack_result.pt", map_location="cpu", weights_only=False)
clean_test = result["clean_test"]
bd_test = result["bd_test"]

def to_numpy_img(x):
    """Tensor/PIL/ndarray 이미지를 matplotlib용 [0,1] numpy 이미지로 변환합니다."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu()
        if x.ndim == 3 and x.shape[0] in [1, 3]:
            x = x.permute(1, 2, 0)
        x = x.numpy()
    x = np.asarray(x)
    if x.max() > 1.5:
        x = x / 255.0
    return np.clip(x, 0, 1)

save_dir = "./record/lf_0_1/report_visual"
os.makedirs(save_dir, exist_ok=True)

n = 8
fig, axes = plt.subplots(3, n, figsize=(2*n, 6))
for i in range(n):
    clean_img, clean_label = clean_test[i]
    bd_img, bd_label = bd_test[i]
    c = to_numpy_img(clean_img)
    b = to_numpy_img(bd_img)
    diff = np.abs(b - c)
    diff = diff / (diff.max() + 1e-8)

    axes[0, i].imshow(c)
    axes[0, i].set_title(f"clean:{clean_label}", fontsize=9)
    axes[1, i].imshow(b)
    axes[1, i].set_title(f"bd:{bd_label}", fontsize=9)
    axes[2, i].imshow(diff)
    axes[2, i].set_title("abs diff", fontsize=9)

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
out = f"{save_dir}/clean_vs_lf_bd_diff.png"
plt.savefig(out, dpi=180)
plt.show()
print("saved:", out)

backup_record()


## 셀 12. FFT 주파수 스펙트럼 비교


In [ ]:
# LF 공격은 주파수 기반 공격이므로 FFT magnitude 차이를 함께 확인합니다.
import numpy as np, matplotlib.pyplot as plt, os

save_dir = "./record/lf_0_1/report_visual"
os.makedirs(save_dir, exist_ok=True)

idx = 0
clean_img, _ = clean_test[idx]
bd_img, _ = bd_test[idx]
c = to_numpy_img(clean_img).mean(axis=2)
b = to_numpy_img(bd_img).mean(axis=2)

def fft_mag(x):
    """이미지를 FFT로 변환한 뒤 보기 쉽게 log magnitude를 반환합니다."""
    f = np.fft.fftshift(np.fft.fft2(x))
    return np.log1p(np.abs(f))

c_fft = fft_mag(c)
b_fft = fft_mag(b)
d_fft = np.abs(b_fft - c_fft)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes[0,0].imshow(c, cmap="gray"); axes[0,0].set_title("clean")
axes[0,1].imshow(b, cmap="gray"); axes[0,1].set_title("LF backdoor")
axes[0,2].imshow(np.abs(b-c), cmap="magma"); axes[0,2].set_title("pixel diff")
axes[1,0].imshow(c_fft, cmap="viridis"); axes[1,0].set_title("clean FFT")
axes[1,1].imshow(b_fft, cmap="viridis"); axes[1,1].set_title("bd FFT")
axes[1,2].imshow(d_fft, cmap="magma"); axes[1,2].set_title("FFT diff")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
out = f"{save_dir}/lf_fft_compare.png"
plt.savefig(out, dpi=180)
plt.show()
print("saved:", out)

backup_record()


## 셀 13. 기준 방어 세트 실행


In [ ]:
# 보고서에서 개별 해석하기 좋은 대표 방어 6종을 먼저 실행합니다.
%cd /content/BackdoorBench

import os

CORE_DEFENSES = [
    ("ft", "./defense/ft.py", "./config/defense/ft/cifar10.yaml"),
    ("nad", "./defense/nad.py", "./config/defense/nad/cifar10.yaml"),
    ("anp", "./defense/anp.py", "./config/defense/anp/cifar10.yaml"),
    ("fp", "./defense/fp.py", "./config/defense/fp/cifar10.yaml"),
    ("abl", "./defense/abl.py", "./config/defense/abl/cifar10.yaml"),
    ("nc", "./defense/nc.py", "./config/defense/nc/cifar10.yaml"),
]

for name, script, yaml_path in CORE_DEFENSES:
    print("\n" + "="*80)
    print("Running defense:", name)
    print("="*80)
    cmd = f"python {script} --result_file lf_0_1 --yaml_path {yaml_path} --dataset cifar10"
    ret = os.system(cmd)
    backup_record()
    if ret != 0:
        print(f"[WARN] {name} failed with exit code {ret}. 다음 방어로 계속 진행합니다.")


## 셀 14. 방어 결과 요약표 자동 수집

In [ ]:
# 방어마다 저장 형식이 조금 달라서, CSV와 defense_result.pt를 최대한 자동 탐색합니다.
import os, glob, torch, pandas as pd

base = "./record/lf_0_1"
rows = []

# 공격 단독 결과를 첫 행에 넣습니다.
attack_df = pd.read_csv(f"{base}/attack_df.csv")
last = attack_df.iloc[-1]
rows.append({
    "method": "attack_only",
    "test_acc": last.get("test_acc", None),
    "test_asr": last.get("test_asr", None),
    "test_ra": last.get("test_ra", None),
    "source": "attack_df.csv"
})

for d in sorted(glob.glob(f"{base}/defense/*")):
    if not os.path.isdir(d):
        continue
    method = os.path.basename(d)
    row = {"method": method, "test_acc": None, "test_asr": None, "test_ra": None, "source": ""}

    csv_candidates = glob.glob(f"{d}/**/*summary*.csv", recursive=True) + glob.glob(f"{d}/**/*df*.csv", recursive=True)
    found = False
    for p in csv_candidates:
        try:
            tmp = pd.read_csv(p)
            cols = set(tmp.columns)
            if {"test_acc", "test_asr", "test_ra"}.issubset(cols):
                r = tmp.iloc[-1]
                row.update({"test_acc": r["test_acc"], "test_asr": r["test_asr"], "test_ra": r["test_ra"], "source": p})
                found = True
                break
        except Exception:
            pass

    if not found:
        pt = f"{d}/defense_result.pt"
        if os.path.exists(pt):
            try:
                obj = torch.load(pt, map_location="cpu", weights_only=False)
                if isinstance(obj, dict):
                    for k in ["test_acc", "acc", "clean_acc"]:
                        if k in obj: row["test_acc"] = obj[k]
                    for k in ["test_asr", "asr"]:
                        if k in obj: row["test_asr"] = obj[k]
                    for k in ["test_ra", "ra"]:
                        if k in obj: row["test_ra"] = obj[k]
                    row["source"] = pt
            except Exception as e:
                row["source"] = f"{pt} load failed: {e}"

    rows.append(row)

result_table = pd.DataFrame(rows)
display(result_table)

out = f"{base}/report_visual/lf_defense_summary_table.csv"
os.makedirs(os.path.dirname(out), exist_ok=True)
result_table.to_csv(out, index=False)
print("saved:", out)
backup_record()

## 셀 15. 방어 비교 막대그래프

In [ ]:
# 공격 단독과 방어별 ACC/ASR/RA를 막대그래프로 비교합니다.
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os

table_path = "./record/lf_0_1/report_visual/lf_defense_summary_table.csv"
table = pd.read_csv(table_path)

plot_df = table.dropna(subset=["test_acc", "test_asr", "test_ra"], how="all").copy()
plot_df = plot_df[["method", "test_acc", "test_asr", "test_ra"]]

fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 0.8), 5))
x = np.arange(len(plot_df))
w = 0.25

ax.bar(x - w, plot_df["test_acc"], width=w, label="Clean ACC")
ax.bar(x, plot_df["test_asr"], width=w, label="ASR")
ax.bar(x + w, plot_df["test_ra"], width=w, label="RA")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["method"], rotation=45, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("score")
ax.set_title("LF Attack and Defense Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out = "./record/lf_0_1/report_visual/lf_defense_bar_chart.png"
plt.savefig(out, dpi=180)
plt.show()
print("saved:", out)
backup_record()

## 셀 16. t-SNE / Grad-CAM / Confusion Matrix

In [ ]:
# 공격 모델과 NAD 방어 모델을 비교 시각화합니다. 다른 방어를 보고 싶으면 defense/nad만 바꾸면 됩니다.
%cd /content/BackdoorBench

# feature space 분포
!python ./analysis/visual_tsne.py \
    --result_file_attack lf_0_1 \
    --result_file_defense None \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset mixed \
    --target_class 0

!python ./analysis/visual_tsne.py \
    --result_file_attack lf_0_1 \
    --result_file_defense lf_0_1/defense/nad \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset mixed \
    --target_class 0

# 모델이 보는 영역 Grad-CAM
!python ./analysis/visual_gradcam.py \
    --result_file_attack lf_0_1 \
    --result_file_defense None \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset bd_test \
    --target_class 0

!python ./analysis/visual_gradcam.py \
    --result_file_attack lf_0_1 \
    --result_file_defense lf_0_1/defense/nad \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset bd_test \
    --target_class 0

# backdoor 입력이 target class로 몰리는지 확인
!python ./analysis/visual_cm.py \
    --result_file_attack lf_0_1 \
    --result_file_defense None \
    --visual_dataset bd_test \
    --target_class 0

!python ./analysis/visual_cm.py \
    --result_file_attack lf_0_1 \
    --result_file_defense lf_0_1/defense/nad \
    --visual_dataset bd_test \
    --target_class 0

backup_record()

## 셀 17. Frequency saliency

In [ ]:
# LF는 주파수 기반 공격이므로 frequency saliency 결과가 보고서 핵심 그림입니다.
%cd /content/BackdoorBench

!python ./analysis/visual_fre.py \
    --result_file_attack lf_0_1 \
    --result_file_defense None \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset mixed \
    --target_class 0

!python ./analysis/visual_fre.py \
    --result_file_attack lf_0_1 \
    --result_file_defense lf_0_1/defense/nad \
    --target_layer_name layer4.1.conv2 \
    --visual_dataset mixed \
    --target_class 0

backup_record()

## 셀 18. 생성 파일 목록 확인

In [ ]:
# 보고서에 넣을 CSV와 이미지 파일 위치를 한 번에 확인합니다.
import glob, os

for p in sorted(glob.glob("./record/lf_0_1/**/*", recursive=True)):
    if p.lower().endswith((".png", ".jpg", ".jpeg", ".pdf", ".csv")):
        print(p)

## 셀 19. 보고서용 Markdown 표 생성

In [ ]:
# 방어 결과 요약표를 Markdown 형식으로 저장합니다.
import pandas as pd, os

table = pd.read_csv("./record/lf_0_1/report_visual/lf_defense_summary_table.csv")
cols = ["method", "test_acc", "test_asr", "test_ra"]
md_table = table[cols].to_markdown(index=False)

out = "./record/lf_0_1/report_visual/report_table.md"
with open(out, "w") as f:
    f.write(md_table)

print(md_table)
print("saved:", out)
backup_record()
